<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Simulate_HCP_to_TMS_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Simulate_HCP_to_TMS_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-Dataset Generalization: HCP-Trained Models → TMS-fMRI Simulation

## Can REST-Trained Models Predict Stimulus Effects in a Different Dataset?

**Objective:**
Use MLP models trained exclusively on **HCP resting-state data** (TR=0.72s) to:
1. Simulate high-temporal-resolution REST and STIM sessions for TMS-fMRI subjects
2. Downsample to match TMS experiment TR (2.0s/2.4s)
3. Compare functional connectivity to empirical TMS-fMRI data

**Key Insight:**
- If Delta FC (Stim - Rest) correlates with empirical despite models trained on *different dataset* and *different TR*
- This proves models learned **generalizable stimulus-response dynamics**
- Not dataset-specific overfitting of TMS responses

**Hypothesis:** HIGH Delta FC correlation → intrinsic network dynamics capture stimulus effects across datasets

## Step 1: Setup and Configuration

In [1]:
# --- Setup and Imports ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys, pickle, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Clone repo + add to path
repo_dir = "/content/BrainStim_ANN_fMRI_HCP"
if not os.path.exists(repo_dir):
    !git clone https://github.com/grabuffo/BrainStim_ANN_fMRI_HCP.git
else:
    print("✓ Repo already exists")

sys.path.append(repo_dir)
from src.preprocessing_tms_fmri import preprocess_run
from src.NPI import build_model, device

print(f"PyTorch device: {device}")

Mounted at /content/drive
Cloning into 'BrainStim_ANN_fMRI_HCP'...
remote: Enumerating objects: 883, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 883 (delta 65), reused 14 (delta 14), pack-reused 740 (from 2)
Receiving objects: 100% (883/883), 99.83 MiB | 12.62 MiB/s, done.
Resolving deltas: 100% (325/325), done.
PyTorch device: cpu


In [2]:
# --- Configuration Parameters ---
ROI_num = 450
using_steps = 3

# TMS experiment TRs
tr_rest = 2.0
tr_stim = 2.4

# HCP simulation: high temporal resolution for accurate stimulus timing
tr_hcp = 0.72  # HCP models trained at this TR
ds_factor = tr_stim / tr_hcp  # Downsampling factor (~3.33)

remove_initial_trs = 12
low_hz, high_hz = 0.008, 0.08

# Simulation parameters
BURN_IN = 30
NOISE_SIGMA = 0.28
STIM_AMP = 10.0
STIM_DURATION_S = tr_stim
RHO_MM = 60.0

rng = np.random.default_rng(42)

# Paths
base_dir = "/content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data"
data_dir = base_dir  # For consistency with other scripts

# HCP data directory for loading models
hcp_preproc_dir = os.path.join(base_dir, "preprocessed_subjects")
hcp_weights_dir = os.path.join(hcp_preproc_dir, "trained_models_MLP")

# TMS data directory for empirical dataset
tms_data_dir = os.path.join(base_dir, "TMS_fMRI")
tms_preproc_dir = os.path.join(base_dir, "preprocessed_subjects_tms_fmri")
dataset_pkl = os.path.join(tms_data_dir, "dataset_tian50_schaefer400_allruns.pkl")

# Output directory for cross-dataset validation
out_dir = os.path.join(tms_preproc_dir, "simulated_from_HCP_to_TMS_validation")
os.makedirs(out_dir, exist_ok=True)

out_pkl = os.path.join(out_dir, "dataset_simulated_HCP_downsampled.pkl")
results_json = os.path.join(out_dir, "simulation_summary_HCP_to_TMS.json")

# Extract cortical ROIs only (Schaefer: indices 50-449, ignore Tian subcortex 0-49)
cortical_roi_indices = np.arange(50, 450)
n_cortical = len(cortical_roi_indices)

print(f"{'='*70}")
print(f"CROSS-DATASET VALIDATION: HCP-Trained → TMS-fMRI")
print(f"{'='*70}")
print(f"\nData paths:")
print(f"  HCP models dir: {hcp_weights_dir}")
print(f"  TMS data dir: {tms_data_dir}")
print(f"  Output dir: {out_dir}")
print(f"\nConfiguration:")
print(f"  ROI_num: {ROI_num}, using_steps: {using_steps}")
print(f"  TR_HCP: {tr_hcp}s (simulation), TR_TMS: {tr_rest}s/{tr_stim}s (target)")
print(f"  Downsampling factor: {ds_factor:.2f}x")
print(f"  Cortical ROIs: {n_cortical} regions (indices {cortical_roi_indices[0]}-{cortical_roi_indices[-1]})")
print(f"  Simulation params: BURN_IN={BURN_IN}, NOISE_SIGMA={NOISE_SIGMA}, STIM_AMP={STIM_AMP}")
print(f"{'='*70}\n")

CROSS-DATASET VALIDATION: HCP-Trained → TMS-fMRI

Data paths:
  HCP models dir: /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/preprocessed_subjects/trained_models_MLP
  TMS data dir: /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/TMS_fMRI
  Output dir: /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/preprocessed_subjects_tms_fmri/simulated_from_HCP_to_TMS_validation

Configuration:
  ROI_num: 450, using_steps: 3
  TR_HCP: 0.72s (simulation), TR_TMS: 2.0s/2.4s (target)
  Downsampling factor: 3.33x
  Cortical ROIs: 400 regions (indices 50-449)
  Simulation params: BURN_IN=30, NOISE_SIGMA=0.28, STIM_AMP=10.0



## Step 2: Load Distance Matrix and Spatial Gaussian Kernel

In [4]:
# --- Load distance matrix for spatial Gaussian kernel ---
try:
    dist_path = os.path.join(tms_data_dir, "atlases", "distance_matrix_450x450_Tian50_Schaefer400.npy")
    D = np.load(dist_path)
    W = np.exp(-(D ** 2) / (2.0 * (RHO_MM ** 2))).astype(np.float32)
    W /= (W[np.arange(ROI_num), np.arange(ROI_num)][:, None] + 1e-8)  # Normalize so target = 1
    print(f"✓ Loaded distance matrix: W shape {W.shape}, range [{W.min():.4f}, {W.max():.4f}]")
    print(f"  Gaussian spread: RHO_MM = {RHO_MM} mm")
except FileNotFoundError as e:
    print(f"⚠ Distance matrix not found: {e}")
    print(f"  Will use direct stimulation only (no spatial spread)")
    W = None

✓ Loaded distance matrix: W shape (450, 450), range [0.0176, 1.0000]
  Gaussian spread: RHO_MM = 60.0 mm


## Step 3: Define Model Architecture and Load Trained Models

In [5]:
# --- Model Architecture: Stimulus-Agnostic MLP ---
# Input: Brain state only (using_steps * ROI_num = 3 * 450 = 1350 dims)
# Hidden: 256 → 128 units with ReLU
# Output: Next predicted ROI activation (450 dims)
# Key: NO stimulus encoding, just intrinsic dynamics

print("✓ Will instantiate models using src.NPI.build_model()")

✓ Will instantiate models using src.NPI.build_model()


In [6]:
# --- Load HCP-trained models (for cross-dataset validation) ---
print("="*70)
print("Loading HCP-trained models (TR=0.72s, NO TMS exposure)")
print("="*70)
print("These models will be used to simulate TMS-fMRI data as a generalization test\n")
print(f"Loading from: {hcp_weights_dir}\n")

trained_models = {}
model_pattern = '_MLP.pt'

# List available model files
if os.path.exists(hcp_weights_dir):
    model_files = [f for f in os.listdir(hcp_weights_dir) if f.endswith(model_pattern)]
    print(f"Found {len(model_files)} HCP model files")
else:
    print(f"⚠ HCP Weights directory not found: {hcp_weights_dir}")
    model_files = []

for model_file in sorted(model_files):
    sub_id = model_file.replace(model_pattern, '')
    model_path = os.path.join(hcp_weights_dir, model_file)

    try:
        torch.serialization.add_safe_globals([torch.nn.modules.linear.Linear, torch.nn.modules.activation.ReLU])
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)

        if isinstance(checkpoint, dict):
            model = build_model("MLP", ROI_num=ROI_num, using_steps=using_steps).to(device)
            model.load_state_dict(checkpoint)
        else:
            model = checkpoint.to(device) if hasattr(checkpoint, 'to') else checkpoint

        model.eval()
        trained_models[sub_id] = model
        print(f"✓ {sub_id} (HCP-trained)")
    except Exception as e:
        print(f"✗ {sub_id} - skipping ({str(e)[:50]}...)")

print(f"\n✅ Loaded {len(trained_models)} HCP-trained models ready for TMS simulation")

Loading HCP-trained models (TR=0.72s, NO TMS exposure)
These models will be used to simulate TMS-fMRI data as a generalization test

Loading from: /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/preprocessed_subjects/trained_models_MLP

Found 10 HCP model files
✓ id_100206 (HCP-trained)
✓ id_100307 (HCP-trained)
✓ id_100408 (HCP-trained)
✓ id_101006 (HCP-trained)
✓ id_101107 (HCP-trained)
✓ id_101309 (HCP-trained)
✓ id_101915 (HCP-trained)
✓ id_102008 (HCP-trained)
✓ id_102109 (HCP-trained)
✓ id_102311 (HCP-trained)

✅ Loaded 10 HCP-trained models ready for TMS simulation


## Step 5: Simulation Helper Functions

In [8]:
# --- Helper Functions for Data Parsing ---

def safe_target_idx(target_vec):
    """Extract target region index from one-hot vector."""
    if target_vec is None:
        return None
    v = np.asarray(target_vec).astype(int).ravel()
    if v.size == 0 or v.sum() != 1:
        return None
    return int(np.argmax(v))

def get_onset_column(df):
    """Find onset column in dataframe."""
    if df is None or len(df) == 0:
        return None
    for col in ["onset", "Onset", "stim_onset", "onset_s", "onset_sec", "time", "t", "seconds"]:
        if col in df.columns:
            return col
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            return col
    return None

def map_onsets_to_steps(onsets_s, tr_model=tr_stim, mode="round"):
    """Map stimulus onsets (seconds) to model steps.

    After removing initial TRs, stimulus times are relative to start of cleaned data.
    This function converts seconds to simulation step indices.
    """
    onsets_s = np.asarray(onsets_s, dtype=float)
    x = onsets_s / float(tr_model)
    if mode == "round":
        steps = np.rint(x).astype(int)
    elif mode == "floor":
        steps = np.floor(x).astype(int)
    else:
        steps = np.ceil(x).astype(int)
    steps = steps[steps >= 0]
    return np.unique(steps)

print("✓ Helper functions defined")

✓ Helper functions defined


In [9]:
# --- Neural Network Inference ---

@torch.no_grad()
def predict_next(model, window_SxN):
    """Predict next state from rolling window of brain state.

    For stimulus-agnostic models:
    - Input: flattened brain state from using_steps (1350 dims for 3 steps × 450 ROIs)
    - Output: next predicted ROI activation (450 dims)
    - Noise is added to output for stochasticity
"""
    brain_state = window_SxN.reshape(-1).astype(np.float32)
    noise = NOISE_SIGMA * rng.normal(0.0, 1.0, size=brain_state.shape).astype(np.float32)
    brain_state = brain_state + noise

    x = torch.tensor(brain_state[None, :], dtype=torch.float32, device=device)
    y = model(x)
    return y.detach().cpu().numpy().squeeze(0)

def simulate_run(model, init_window_SxN, n_steps, stim_steps=None, target_idx=None, W=None, stim_amplitude=None):
    """Simulate brain activity time series with optional TMS stimulation.

    The model learns intrinsic dynamics during normal brain function.
    When we apply stimulation, we perturb the state at the target region using:
    - Spatial Gaussian kernel (W) to spread stimulation across regions
    - Empirical stimulus timing (stim_steps) to apply perturbations at correct times
"""
    init_window_SxN = np.asarray(init_window_SxN, dtype=np.float32)
    assert init_window_SxN.shape == (using_steps, ROI_num)

    if stim_amplitude is None:
        stim_amplitude = STIM_AMP

    if stim_steps is not None:
        stim_steps = set(int(s) for s in stim_steps)
    else:
        stim_steps = set()

    do_stim = (target_idx is not None) and (len(stim_steps) > 0)
    w = init_window_SxN.copy()

    # Burn-in phase: let model settle to attractor without stimulation
    for _ in range(BURN_IN):
        y = predict_next(model, w)
        w = np.vstack([w[1:], y[None, :]])

    # Simulate with optional spatial TMS
    out = np.zeros((n_steps, ROI_num), dtype=np.float32)
    for t in range(n_steps):
        if do_stim and (t in stim_steps):
            if W is not None:
                w[-1, :] += stim_amplitude * W[target_idx, :]
            else:
                w[-1, target_idx] += stim_amplitude

        y = predict_next(model, w)
        out[t] = y
        w = np.vstack([w[1:], y[None, :]])

    return out

print("✓ Simulation functions defined")

✓ Simulation functions defined


## Step 6: Functional Connectivity Computation

In [10]:
# --- Functional Connectivity Functions ---

def compute_fc_matrix(time_series):
    """Compute functional connectivity (Pearson correlation) matrix."""
    ts_std = (time_series - time_series.mean(axis=0)) / (time_series.std(axis=0) + 1e-8)
    fc = np.corrcoef(ts_std.T)
    return fc

def extract_upper_triangle(fc_matrix):
    """Extract upper triangle of FC matrix (excluding diagonal) as vector."""
    N = fc_matrix.shape[0]
    indices = np.triu_indices(N, k=1)
    return fc_matrix[indices]

def downsample_timeseries(ts_hires, factor):
    """Downsample high-resolution time series to lower resolution."""
    from scipy.signal import decimate

    factor_int = int(np.round(factor))
    if factor_int < 2:
        return ts_hires

    ts_down = np.zeros((ts_hires.shape[0] // factor_int, ts_hires.shape[1]), dtype=np.float32)
    for i in range(ts_hires.shape[1]):
        ts_down[:, i] = decimate(ts_hires[:, i], factor_int, zero_phase=True)
    return ts_down

print("✓ FC computation functions defined")

✓ FC computation functions defined


## Step 4: Load Empirical Dataset and Group by Target

In [11]:
# --- Load empirical dataset ---
print(f"Loading empirical dataset from {dataset_pkl}...")
with open(dataset_pkl, "rb") as f:
    dataset_emp = pickle.load(f)

print(f"✓ Loaded {len(dataset_emp)} subjects")

# --- SELECT ONE HCP MODEL FOR CROSS-DATASET VALIDATION ---
reference_hcp_model = list(trained_models.values())[0]
reference_hcp_sub = list(trained_models.keys())[0]

print(f"\n{'='*70}")
print(f"CROSS-DATASET VALIDATION SETUP:")
print(f"{'='*70}")
print(f"Selected HCP-trained model: {reference_hcp_sub}")
print(f"Target population: {len(dataset_emp)} TMS-fMRI subjects\n")
print(f"Approach: Target-based comparison")
print(f"  1. Group empirical data by stimulation TARGET")
print(f"  2. Simulate all TMS subjects with single HCP model")
print(f"  3. Compare target-averaged FC (empirical vs simulated)")
print(f"{'='*70}\n")

# --- GROUP EMPIRICAL DATA BY STIMULATION TARGET ---
print("Parsing empirical TMS-fMRI data by target...")
empirical_by_target = {}  # target_idx -> list of {sub_id, rest_fc, stim_fc, delta_fc}

for sub_id in sorted(dataset_emp.keys()):
    if 'task-rest' not in dataset_emp[sub_id] or 'task-stim' not in dataset_emp[sub_id]:
        continue

    # Compute mean REST FC across all rest runs for this subject
    rest_fc_list = []
    for run_idx, run_data in dataset_emp[sub_id]['task-rest'].items():
        ts = run_data.get('time series')
        if ts is not None and ts.shape[0] > remove_initial_trs:
            ts_proc = preprocess_run(ts[remove_initial_trs:], tr=tr_rest,
                                     low=low_hz, high=high_hz, order=2, zscore=True)
            if ts_proc.shape[0] > using_steps:
                rest_fc_list.append(compute_fc_matrix(ts_proc[:, cortical_roi_indices]))

    if len(rest_fc_list) == 0:
        continue

    mean_rest_fc = np.mean(rest_fc_list, axis=0)

    # For each stim run, extract target and compute FC
    for run_idx, run_data in dataset_emp[sub_id]['task-stim'].items():
        ts = run_data.get('time series')
        target_vec = run_data.get('target')

        if ts is None or target_vec is None or ts.shape[0] <= remove_initial_trs:
            continue

        target_idx = safe_target_idx(target_vec)
        if target_idx is None:
            continue

        ts_proc = preprocess_run(ts[remove_initial_trs:], tr=tr_stim,
                                 low=low_hz, high=high_hz, order=2, zscore=True)

        if ts_proc.shape[0] <= using_steps:
            continue

        stim_fc = compute_fc_matrix(ts_proc[:, cortical_roi_indices])
        delta_fc = stim_fc - mean_rest_fc

        if target_idx not in empirical_by_target:
            empirical_by_target[target_idx] = []

        empirical_by_target[target_idx].append({
            'sub_id': sub_id,
            'rest_fc': mean_rest_fc,
            'stim_fc': stim_fc,
            'delta_fc': delta_fc
        })

print(f"✓ Grouped empirical data by {len(empirical_by_target)} unique targets")
for target_idx in sorted(empirical_by_target.keys()):
    n_entries = len(empirical_by_target[target_idx])
    n_subs = len(set(e['sub_id'] for e in empirical_by_target[target_idx]))
    print(f"  Target {target_idx}: {n_entries} stim sessions from {n_subs} subjects")

Loading empirical dataset from /content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data/TMS_fMRI/dataset_tian50_schaefer400_allruns.pkl...


/tmp/ipykernel_178/4220225774.py:4: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  dataset_emp = pickle.load(f)


✓ Loaded 46 subjects

CROSS-DATASET VALIDATION SETUP:
Selected HCP-trained model: id_100206
Target population: 46 TMS-fMRI subjects

Approach: Target-based comparison
  1. Group empirical data by stimulation TARGET
  2. Simulate all TMS subjects with single HCP model
  3. Compare target-averaged FC (empirical vs simulated)

Parsing empirical TMS-fMRI data by target...
✓ Grouped empirical data by 11 unique targets
  Target 155: 42 stim sessions from 40 subjects
  Target 220: 28 stim sessions from 27 subjects
  Target 231: 44 stim sessions from 42 subjects
  Target 305: 43 stim sessions from 41 subjects
  Target 342: 41 stim sessions from 39 subjects
  Target 359: 43 stim sessions from 41 subjects
  Target 366: 37 stim sessions from 36 subjects
  Target 386: 30 stim sessions from 29 subjects
  Target 392: 35 stim sessions from 33 subjects
  Target 401: 45 stim sessions from 43 subjects
  Target 403: 44 stim sessions from 41 subjects


## Step 7: Generate Simulated Dataset by Target

In [12]:
# --- Generate simulated dataset by target using single HCP model ---
print("="*70)
print("GENERATING SIMULATED DATASET: HCP→TMS Cross-Dataset Validation")
print("="*70)
print(f"\nSimulation strategy:")
print(f"  Model: {reference_hcp_sub} (HCP-trained, no TMS exposure)")
print(f"  Subjects: All {len(dataset_emp)} TMS-fMRI subjects")
print(f"  Approach: Target-based simulation and comparison")
print(f"\nTechnical details:")
print(f"  1. Simulate at HCP resolution (TR={tr_hcp}s) for precise stimulus timing")
print(f"  2. Downsample {ds_factor:.2f}x to match TMS TR ({tr_rest}s/{tr_stim}s)")
print(f"  3. Group results by stimulation target")
print(f"  4. Compare target-averaged FC (empirical vs simulated)")
print()

simulated_by_target = {}  # target_idx -> list of {sub_id, rest_fc, stim_fc, delta_fc}
sim_errors = []

# For each empirical target, simulate responses with HCP model
for target_idx in sorted(empirical_by_target.keys()):
    print(f"\nProcessing Target {target_idx}...")
    simulated_by_target[target_idx] = []

    for emp_entry in empirical_by_target[target_idx]:
        sub_id = emp_entry['sub_id']

        if 'task-rest' not in dataset_emp[sub_id] or 'task-stim' not in dataset_emp[sub_id]:
            continue

        try:
            # ---- COLLECT REST INITIALIZATION FROM REST ----
            rest_fc_list = []
            rest_init_window = None

            for run_idx, run_data in dataset_emp[sub_id]['task-rest'].items():
                ts = run_data.get('time series')
                if ts is None or ts.shape[0] <= remove_initial_trs:
                    continue

                ts_proc = preprocess_run(ts[remove_initial_trs:], tr=tr_rest,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] > using_steps:
                    rest_fc_list.append(compute_fc_matrix(ts_proc[:, cortical_roi_indices]))
                    if rest_init_window is None:
                        rest_init_window = ts_proc[:using_steps].copy()

            if rest_init_window is None or len(rest_fc_list) == 0:
                continue

            mean_rest_fc = np.mean(rest_fc_list, axis=0)

            # ---- FIND MATCHING STIM RUN FOR THIS TARGET ----
            for run_idx, run_data in dataset_emp[sub_id]['task-stim'].items():
                ts_emp = run_data.get('time series')
                target_vec = run_data.get('target')
                stim_time_df = run_data.get('stim time')

                if ts_emp is None or target_vec is None:
                    continue

                if safe_target_idx(target_vec) != target_idx:
                    continue

                if ts_emp.shape[0] <= remove_initial_trs:
                    continue

                # ---- PREPROCESS and EXTRACT STIMULUS TIMING ----
                ts_proc = preprocess_run(ts_emp[remove_initial_trs:], tr=tr_stim,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] <= using_steps:
                    continue

                # Extract stimulus timing in HIGH-RESOLUTION steps
                stim_steps_hires = []
                if stim_time_df is not None and len(stim_time_df) > 0:
                    onset_col = get_onset_column(stim_time_df)
                    if onset_col is not None:
                        onsets_s = stim_time_df[onset_col].values
                        onsets_s = onsets_s[onsets_s >= remove_initial_trs * tr_stim]
                        onsets_s = onsets_s - remove_initial_trs * tr_stim
                        stim_steps_hires = map_onsets_to_steps(onsets_s, tr_model=tr_hcp)

                if len(stim_steps_hires) == 0:
                    continue

                # Calculate HCP-resolution steps needed
                n_tms_steps = ts_proc.shape[0]
                n_hcp_steps = int(np.ceil(n_tms_steps * ds_factor))

                # Simulate with HCP model at high resolution
                sim_ts_hires = simulate_run(reference_hcp_model, rest_init_window, n_hcp_steps,
                                           stim_steps=stim_steps_hires,
                                           target_idx=target_idx, W=W)

                # Downsample to TMS TR
                sim_ts_downsampled = downsample_timeseries(sim_ts_hires, ds_factor)[:n_tms_steps]

                # Compute simulated FC
                sim_stim_fc = compute_fc_matrix(sim_ts_downsampled[:, cortical_roi_indices])
                sim_delta_fc = sim_stim_fc - mean_rest_fc

                # Store result
                simulated_by_target[target_idx].append({
                    'sub_id': sub_id,
                    'rest_fc': mean_rest_fc,
                    'stim_fc': sim_stim_fc,
                    'delta_fc': sim_delta_fc
                })
                break  # Only need first matching stim run for this target

        except Exception as e:
            sim_errors.append(f"{sub_id} target {target_idx}: {str(e)[:60]}")

    n_sims = len(simulated_by_target[target_idx])
    print(f"  ✓ Simulated {n_sims} subjects for Target {target_idx}")

print("\n" + "="*70)
print(f"✓ Simulation complete: {len(simulated_by_target)} targets processed")
for target_idx in sorted(simulated_by_target.keys()):
    n = len(simulated_by_target[target_idx])
    print(f"  Target {target_idx}: {n} simulated responses")
print("="*70)

if sim_errors:
    print(f"\n⚠ {len(sim_errors)} simulation errors encountered:")
    for err in sim_errors[:3]:
        print(f"  {err}")
    if len(sim_errors) > 3:
        print(f"  ... and {len(sim_errors)-3} more")

GENERATING SIMULATED DATASET: HCP→TMS Cross-Dataset Validation

Simulation strategy:
  Model: id_100206 (HCP-trained, no TMS exposure)
  Subjects: All 46 TMS-fMRI subjects
  Approach: Target-based simulation and comparison

Technical details:
  1. Simulate at HCP resolution (TR=0.72s) for precise stimulus timing
  2. Downsample 3.33x to match TMS TR (2.0s/2.4s)
  3. Group results by stimulation target
  4. Compare target-averaged FC (empirical vs simulated)


Processing Target 155...
  ✓ Simulated 42 subjects for Target 155

Processing Target 220...
  ✓ Simulated 28 subjects for Target 220

Processing Target 231...
  ✓ Simulated 44 subjects for Target 231

Processing Target 305...
  ✓ Simulated 43 subjects for Target 305

Processing Target 342...
  ✓ Simulated 41 subjects for Target 342

Processing Target 359...
  ✓ Simulated 43 subjects for Target 359

Processing Target 366...
  ✓ Simulated 37 subjects for Target 366

Processing Target 386...
  ✓ Simulated 30 subjects for Target 386



## Step 8: Target-Based Validation & Comparison

In [13]:
# --- Compute target-averaged FC and compare empirical vs simulated ---
print("="*70)
print("TARGET-AVERAGED FC COMPARISON: Empirical vs HCP-Simulated")
print("="*70)
print("Computing mean FC per target across all subjects...\n")

target_comparison = {}  # target_idx -> {'r_rest', 'r_stim', 'r_delta', 'n_subjects'}

for target_idx in sorted(empirical_by_target.keys()):
    if target_idx not in simulated_by_target or len(simulated_by_target[target_idx]) == 0:
        print(f"Target {target_idx}: No simulated data, skipping")
        continue

    emp_entries = empirical_by_target[target_idx]
    sim_entries = simulated_by_target[target_idx]

    # Ensure we have matching subjects
    emp_subs = set(e['sub_id'] for e in emp_entries)
    sim_subs = set(e['sub_id'] for e in sim_entries)
    common_subs = emp_subs & sim_subs

    if len(common_subs) == 0:
        print(f"Target {target_idx}: No matching subjects between empirical and simulated")
        continue

    # Average FC matrices across common subjects
    emp_rest_fc_list = [e['rest_fc'] for e in emp_entries if e['sub_id'] in common_subs]
    emp_stim_fc_list = [e['stim_fc'] for e in emp_entries if e['sub_id'] in common_subs]
    emp_delta_fc_list = [e['delta_fc'] for e in emp_entries if e['sub_id'] in common_subs]

    sim_rest_fc_list = [s['rest_fc'] for s in sim_entries if s['sub_id'] in common_subs]
    sim_stim_fc_list = [s['stim_fc'] for s in sim_entries if s['sub_id'] in common_subs]
    sim_delta_fc_list = [s['delta_fc'] for s in sim_entries if s['sub_id'] in common_subs]

    # Mean FC matrices
    emp_mean_rest = np.mean(emp_rest_fc_list, axis=0)
    emp_mean_stim = np.mean(emp_stim_fc_list, axis=0)
    emp_mean_delta = np.mean(emp_delta_fc_list, axis=0)

    sim_mean_rest = np.mean(sim_rest_fc_list, axis=0)
    sim_mean_stim = np.mean(sim_stim_fc_list, axis=0)
    sim_mean_delta = np.mean(sim_delta_fc_list, axis=0)

    # Extract upper triangles and compute correlations
    emp_rest_vec = extract_upper_triangle(emp_mean_rest)
    emp_stim_vec = extract_upper_triangle(emp_mean_stim)
    emp_delta_vec = extract_upper_triangle(emp_mean_delta)

    sim_rest_vec = extract_upper_triangle(sim_mean_rest)
    sim_stim_vec = extract_upper_triangle(sim_mean_stim)
    sim_delta_vec = extract_upper_triangle(sim_mean_delta)

    # Compute Pearson correlations
    r_rest = np.corrcoef(emp_rest_vec, sim_rest_vec)[0, 1]
    r_stim = np.corrcoef(emp_stim_vec, sim_stim_vec)[0, 1]
    r_delta = np.corrcoef(emp_delta_vec, sim_delta_vec)[0, 1]

    target_comparison[target_idx] = {
        'r_rest': float(r_rest) if not np.isnan(r_rest) else 0.0,
        'r_stim': float(r_stim) if not np.isnan(r_stim) else 0.0,
        'r_delta': float(r_delta) if not np.isnan(r_delta) else 0.0,
        'n_subjects': len(common_subs),
        'subjects': list(common_subs)
    }

# Display results
print("\n" + "="*70)
print("TARGET-AVERAGED FC CORRELATION RESULTS (CORTICAL ONLY)")
print("="*70)
print(f"{'Target':<10} {'r_rest':<12} {'r_stim':<12} {'r_delta':<12} {'N_subj':<8}")
print("-"*60)

for target_idx in sorted(target_comparison.keys()):
    res = target_comparison[target_idx]
    print(f"Target {target_idx:<2} {res['r_rest']:>10.4f}   {res['r_stim']:>10.4f}   {res['r_delta']:>10.4f}   {res['n_subjects']:>6}")

# Summary statistics
if target_comparison:
    all_r_rest = [v['r_rest'] for v in target_comparison.values()]
    all_r_stim = [v['r_stim'] for v in target_comparison.values()]
    all_r_delta = [v['r_delta'] for v in target_comparison.values()]

    print("-"*60)
    print(f"Mean     {np.mean(all_r_rest):>10.4f}   {np.mean(all_r_stim):>10.4f}   {np.mean(all_r_delta):>10.4f}")
    print(f"Std      {np.std(all_r_rest):>10.4f}   {np.std(all_r_stim):>10.4f}   {np.std(all_r_delta):>10.4f}")
    print(f"Min      {np.min(all_r_rest):>10.4f}   {np.min(all_r_stim):>10.4f}   {np.min(all_r_delta):>10.4f}")
    print(f"Max      {np.max(all_r_rest):>10.4f}   {np.max(all_r_stim):>10.4f}   {np.max(all_r_delta):>10.4f}")

print("="*70)
print("\nInterpretation:")
print("  r_rest: Can HCP dynamics match empirical REST FC? (structural learning)")
print("  r_stim: Can HCP dynamics match empirical STIM FC? (stimulus response)")
print("  r_delta: Can HCP dynamics capture FC CHANGE due to stimulation?")
print("="*70)

TARGET-AVERAGED FC COMPARISON: Empirical vs HCP-Simulated
Computing mean FC per target across all subjects...


TARGET-AVERAGED FC CORRELATION RESULTS (CORTICAL ONLY)
Target     r_rest       r_stim       r_delta      N_subj  
------------------------------------------------------------
Target 155     1.0000       0.2414       0.2336       40
Target 220     1.0000       0.2834       0.2253       27
Target 231     1.0000       0.2837       0.2886       42
Target 305     1.0000       0.4329       0.3112       41
Target 342     1.0000       0.4066       0.1726       39
Target 359     1.0000       0.3706       0.2714       41
Target 366     1.0000       0.2726       0.2930       36
Target 386     1.0000       0.4511       0.2183       29
Target 392     1.0000       0.3284       0.1588       33
Target 401     1.0000       0.3703       0.2002       43
Target 403     1.0000       0.3868       0.2326       41
------------------------------------------------------------
Mean         1.0000      

## Summary: Cross-Dataset Validation Results

### What This Notebook Tests

This notebook evaluates whether MLP models trained **exclusively on HCP resting-state data** can generalize to predict **TMS stimulation effects** in an entirely different dataset.

### Experimental Design

1. **Models**: HCP-trained (TR=0.72s, no TMS during training)
2. **Data**: TMS-fMRI empirical recordings (TR=2.0-2.4s, includes stimulation)
3. **Validation**: Target-based comparison after cross-TR downsampling

### Expected Outcomes

- **HIGH r_rest**: Models successfully captured REST network structure → good generalization of intrinsic dynamics
- **MODERATE r_stim**: Models respond to stimulation even though untrained on TMS → indicates genuine stimulus sensitivity
- **HIGH r_delta**: Models capture stimulus-induced FC changes → proves learned dynamics are stimulus-responsive

### Interpretation Guide

- **If r_delta > r_stim**: Model learned the *effect signature* of stimulation (GOOD)
- **If all correlations are low**: Models may need stronger anatomical constraints or attention mechanisms
- **If r_rest is high but r_delta is low**: Models capture structure but miss stimulus integration